#Dati con fattore comune
genera dati casuali, una volta analizzati restituisce media delle correlazioni degli asset casuali e la deviazione standard per ciascun asset
#Rischio al crescere di n
analisi del rapporto rischio-numero asset e confronto con la formula teorica del rischio
#Crescita del capitale
analisi della crescita del capitale in un portafoglio equipesato, riportando il capitale finale, massimo, minimo e il relativo rendimento del portafoglio
#Monte Carlo
analisi di 500 generazioni di dati diverse con stesso shape analizzando i capitali finali alla fine di ogni simulazione trovandone il massimo, minimo, media, deviazione standard e percentuale di generazioni con capitale finale inferiore a quello di partenza

In [16]:
import numpy as np
#COSTANTI
N_ASSET = 50
N_GIORNI = 250
MU_MERCATO = 0.0005
SIGMA_MERCATO = 0.007
SIGMA_IDIO = 0.007
CAPITALE_INIZIALE = 10000
N_SIMULAZIONI = 500
#Dati con fattore comune======================
rng = np.random.default_rng(42)
mercato = rng.normal(MU_MERCATO, SIGMA_MERCATO, (1, N_GIORNI))
idiosincratico = rng.normal(0, SIGMA_IDIO, (N_ASSET,N_GIORNI))
rendimenti = mercato + idiosincratico
matrice = np.corrcoef(rendimenti)
matrice_ricerca = np.copy(matrice)
np.fill_diagonal(matrice_ricerca, np.nan)
media_correlazioni = np.nanmean(matrice_ricerca)
deviazione_per_asset = np.std(rendimenti, axis=1)
print(f"Media delle correlazioni: {media_correlazioni:.2f}")
deviazione_media_per_asset = np.mean(deviazione_per_asset)
print(f"Deviazione media per asset: {deviazione_media_per_asset*100:.2f}%")
deviazione_massima = np.max(deviazione_per_asset)
print(f"Deviazione massima: {deviazione_massima*100:.2f}")
deviazione_minima = np.min(deviazione_per_asset)
print(f"Deviazione minima: {deviazione_minima*100:.2f}")
#Rischio al crescere di n ==========================
n = [1, 5, 20, 100, 500]
for x in n:
  idiosincratico_simulazione_rischio = rng.normal(0, SIGMA_IDIO, (x,N_GIORNI))
  rendimenti_simulazione_rischio = mercato + idiosincratico_simulazione_rischio
  media_rendimenti_simulazione_rischio = np.mean(rendimenti_simulazione_rischio, axis=0)
  deviazione_standard_simulazione_rischio = np.std(media_rendimenti_simulazione_rischio)*100
  dev_std_teorica = np.sqrt(SIGMA_MERCATO**2 + SIGMA_IDIO**2 /x)
  print(f"Deviazione standard {x} asset: {deviazione_standard_simulazione_rischio:.2f}%")
  print(f"Deviazione standard teorica: {dev_std_teorica*100:.2f}%")
#Crescita del capitale=============================
rendimento_medio_giornaliero = np.mean(rendimenti, axis=0)
capitale_storico = CAPITALE_INIZIALE * np.cumprod(rendimento_medio_giornaliero+1)
print(f"Capitale finale: {capitale_storico[-1]:.2f}")
capitale_massimo = np.max(capitale_storico)
capitale_minimo = np.min(capitale_storico)
print(f"Massimo capitale raggiunto: {capitale_massimo:.2f}")
print(f"Minimo capitale raggiunto: {capitale_minimo:.2f}")
rendimento_totale_simulazione = capitale_storico[-1]/CAPITALE_INIZIALE-1
print(f"Totale rendimento portafoglio: {rendimento_totale_simulazione*100:.2f}%")
#Monte Carlo=========================
capitali_finali_monte_carlo =  []
for i in range(500):
  rng_simulazione_monte_carlo = np.random.default_rng(i)
  mercato_sim = rng_simulazione_monte_carlo.normal(0.0005, 0.007, (1, 250))
  idiosincratico_sim = rng_simulazione_monte_carlo.normal(0, 0.007, (50, 250))
  rendimenti_sim = mercato_sim + idiosincratico_sim
  rend_portafoglio = np.mean(rendimenti_sim, axis=0)
  capitale_storico_monte_carlo = CAPITALE_INIZIALE * np.cumprod(rend_portafoglio+1)
  capitali_finali_monte_carlo.append(capitale_storico_monte_carlo[-1])
capitali_finali_monte_carlo = np.array(capitali_finali_monte_carlo)
media_capitali_finali_monte_carlo = np.mean(capitali_finali_monte_carlo)
dev_capitali_finali = np.std(capitali_finali_monte_carlo)
capitale_finale_monte_carlo_max = np.max(capitali_finali_monte_carlo)
capitale_finale_monte_carlo_min = np.min(capitali_finali_monte_carlo)
print(f"Media capitali finali: {media_capitali_finali_monte_carlo:.2f}")
print(f"Deviazione standard capitali finali: {dev_capitali_finali:.2f}")
print(f"Massimo capitale finale raggiunto: {capitale_finale_monte_carlo_max:.2f}")
print(f"Minimo capitale finale raggiunto: {capitale_finale_monte_carlo_min:.2f}")
percentuale_valori_finali_sotto_10000 = np.mean(capitali_finali_monte_carlo < 10000)
print(f"Percentuale valori finali sotto 10000: {percentuale_valori_finali_sotto_10000*100:.2f}%")
#verifica
print("==================VERIFICA==================")
print(round(mercato.std()*100,2))
print(round(np.median(capitali_finali_monte_carlo),2))

Media delle correlazioni: 0.47
Deviazione media per asset: 0.96%
Deviazione massima: 1.02
Deviazione minima: 0.89
Deviazione standard 1 asset: 0.97%
Deviazione standard teorica: 0.99%
Deviazione standard 5 asset: 0.72%
Deviazione standard teorica: 0.77%
Deviazione standard 20 asset: 0.66%
Deviazione standard teorica: 0.72%
Deviazione standard 100 asset: 0.66%
Deviazione standard teorica: 0.70%
Deviazione standard 500 asset: 0.66%
Deviazione standard teorica: 0.70%
Capitale finale: 10135.35
Massimo capitale raggiunto: 10718.98
Minimo capitale raggiunto: 9813.83
Totale rendimento portafoglio: 1.35%
Media capitali finali: 11439.14
Deviazione standard capitali finali: 1350.71
Massimo capitale finale raggiunto: 16364.06
Minimo capitale finale raggiunto: 7615.83
Percentuale valori finali sotto 10000: 14.60%
==================VERIFICA==================
0.66
11401.43


In [ ]:
#ESERCIZIO PREPARAZIONE
import numpy as np
r = np.array([0.10, -0.10, 0.05, 0.02])
capitale_iniziale = 100
print(np.cumprod(r))
lista_capitali_per_periodo = []
capitale_iniziale_simulazione = capitale_iniziale
for rendimento in r:
    capitale_iniziale_simulazione = capitale_iniziale_simulazione+(capitale_iniziale_simulazione*rendimento)
    lista_capitali_per_periodo.append(float(capitale_iniziale_simulazione))
print(lista_capitali_per_periodo)
capitale_finale = capitale_iniziale+(capitale_iniziale * np.sum(r))
print(f"Capitale finale con 7%: {capitale_finale:.2f}")

[ 1.e-01 -1.e-02 -5.e-04 -1.e-05]
[110.0, 99.0, 103.95, 106.029]
Capitale finale con 7%: 107.00
